# EDA

- 0813

### 년도별로 분리

In [ ]:
# 연도별로 분리

import os
import re
import pandas as pd


df = pd.read_csv("../../01-4_year-seperation/per_100k_data.csv", encoding="utf-8-sig")

suffixes = []
pattern = re.compile(r'(\d{2})$')

for col in df.columns:
    m = pattern.search(col)
    if m:
        suffixes.append(m.group(1))

suffixes = sorted(set(suffixes))

# 두 자리 숫자 -> 연도(2000+xx)로 매핑 (예: 20 -> 2020)
year_map = {sfx: 2000 + int(sfx) for sfx in suffixes}

# 저장 폴더
output_dir = '../../01-4_year-seperation'
os.makedirs(output_dir, exist_ok=True)

saved_files = []

# 년도 컬럼 외의 "공통 컬럼(뒤에 숫자 없는 컬럼)" 유지
base_cols = [c for c in df.columns if not pattern.search(c)]

for sfx in suffixes:
    year = year_map[sfx]
    year_cols = [c for c in df.columns if c.endswith(sfx)]
    clean_year_cols = [re.sub(r'[_\-]?\d{2}$', '', c) for c in year_cols]
    out_df = pd.concat([df[base_cols], df[year_cols]], axis=1)
    out_df.columns = base_cols + clean_year_cols
    out_path = os.path.join(output_dir, f"{year}.csv")
    out_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    saved_files.append(out_path)

### 년도 컬럼 생성해서 모든 값 해당 년도 넣고 정리

In [ ]:
years = ['2020', '2021', '2022', '2023']

data_by_year = {}
for year in years:
    df = pd.read_csv(f"../../01-4_year-seperation/{year}.csv", encoding="utf-8-sig")
    df.insert(0, "YEAR", int(year))
    
    cols = ["SGG_CODE", "YEAR"] + [c for c in df.columns if c not in ["sgg_Code", "YEAR"]]
    df = df[cols]
    
    df.to_csv(f"../../01-4_year-seperation/{year}.csv", index=False, encoding="utf-8-sig")

### 년도 합치기

In [ ]:
import pandas as pd

df_20 = pd.read_csv("../../01-4_year-seperation/2020.csv", encoding="utf-8-sig")
df_21 = pd.read_csv("../../01-4_year-seperation/2021.csv", encoding="utf-8-sig")
df_22 = pd.read_csv("../../01-4_year-seperation/2022.csv", encoding="utf-8-sig")
df_23 = pd.read_csv("../../01-4_year-seperation/2023.csv", encoding="utf-8-sig")

df_add = pd.concat([df_20, df_21, df_22, df_23])
df_add.drop(columns=["SGG_CODE.1"], inplace=True)
# print(df_add)
df_add.to_csv("../../01-4_year-seperation/data_add.csv", index=False, encoding="utf-8-sig")

### 종속변수: (전출 - 전입) / 연앙인구

In [ ]:
# 일단 종속변수: YOUTH_NET_MOVE_RATE (청년(15~34) 순이동률)
# 이름 알기 쉽게 Y로 변경하고 컬럼 제일 오른쪽으로 배치
df = pd.read_csv("../../01-4_year-seperation/data_add.csv", encoding="utf-8-sig")
rename_df = df.rename(columns={"YOUTH_NET_MOVE_RATE": "Y"})

# 종속변수(Y) 컬럼 제일 오른쪽에 배치
new_order = [c for c in rename_df.columns if c != "Y"] + ["Y"]
rename_df = rename_df[new_order]

rename_df.to_csv("../../01-4_year-seperation/data_add.csv", index=False, encoding="utf-8-sig")